In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix

In [2]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
from google.colab import files

uploaded = files.upload()

Saving archive (2).zip to archive (2).zip


In [4]:
import zipfile

with zipfile.ZipFile('archive (2).zip', 'r') as zip_ref:
    zip_ref.extractall('dataset')

In [5]:
import os

for root, dirs, files in os.walk('dataset'):
    for file in files:
        print(os.path.join(root, file))

dataset/Genre Classification Dataset/test_data_solution.txt
dataset/Genre Classification Dataset/train_data.txt
dataset/Genre Classification Dataset/description.txt
dataset/Genre Classification Dataset/test_data.txt


In [6]:
import pandas as pd

df = pd.read_csv(
    'dataset/Genre Classification Dataset/train_data.txt',
    sep=' ::: ',
    names=['ID', 'TITLE', 'GENRE', 'DESCRIPTION'],
    engine='python'
)

df.head()

,ID,TITLE,GENRE,DESCRIPTION
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his doc...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous re...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fiel...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends meet...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-rec...


In [7]:
print(df.columns)

Index(['ID', 'TITLE', 'GENRE', 'DESCRIPTION'], dtype='object')


In [8]:
print(df.shape)

(54214, 4)


In [9]:
print(df.isnull().sum())

ID             0
TITLE          0
GENRE          0
DESCRIPTION    0
dtype: int64


In [10]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()

    words = [stemmer.stem(word)
             for word in words
             if word not in stop_words]

    return " ".join(words)

In [11]:
df['clean_description'] = df['DESCRIPTION'].apply(clean_text)

In [12]:
df[['DESCRIPTION', 'clean_description']].head()

,DESCRIPTION,clean_description
0,Listening in to a conversation between his doc...,listen convers doctor parent yearold oscar lea...
1,A brother and sister with a past incestuous re...,brother sister past incestu relationship curre...
2,As the bus empties the students for their fiel...,bu empti student field trip museum natur histo...
3,To help their unemployed father make ends meet...,help unemploy father make end meet edith twin ...
4,The film's title refers not only to the un-rec...,film titl refer unrecov bodi ground zero also ...


In [13]:
X = df['clean_description']
y = df['GENRE']

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_tfidf = tfidf.fit_transform(X)

print(X_tfidf.shape)

(54214, 10000)


In [14]:
print(df.columns)

Index(['ID', 'TITLE', 'GENRE', 'DESCRIPTION', 'clean_description'], dtype='object')


In [15]:
print(X_tfidf.shape)

(54214, 10000)


In [16]:
df.drop_duplicates(inplace=True)

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
print(X_train.shape)
print(X_test.shape)

(43371,)
(10843,)


In [19]:
model = LogisticRegression(max_iter=1000)

model.fit(tfidf.transform(X_train), y_train)
y_pred = model.predict(tfidf.transform(X_test))

In [20]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.583971225675551


In [21]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

      action       0.53      0.28      0.36       263
       adult       0.81      0.32      0.46       118
   adventure       0.58      0.16      0.25       155
   animation       0.41      0.07      0.12       100
   biography       0.00      0.00      0.00        53
      comedy       0.53      0.60      0.56      1490
       crime       0.30      0.03      0.05       101
 documentary       0.67      0.85      0.75      2619
       drama       0.54      0.78      0.64      2723
      family       0.48      0.08      0.13       157
     fantasy       0.00      0.00      0.00        65
   game-show       0.87      0.51      0.65        39
     history       0.00      0.00      0.00        49
      horror       0.65      0.57      0.61       441
       music       0.68      0.43      0.53       146
     musical       0.50      0.02      0.04        55
     mystery       0.33      0.02      0.03        64
        news       1.00    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
confusion_matrix(y_test, y_pred)

array([[  73,    0,    2,    0,    0,   25,    1,   28,   89,    0,    0,
           0,    0,   11,    0,    0,    0,    0,    0,    0,    2,    8,
           9,    0,   12,    0,    3],
       [   0,   38,    9,    0,    0,   33,    0,    5,   20,    0,    0,
           0,    0,    1,    1,    0,    0,    0,    0,    0,    0,   11,
           0,    0,    0,    0,    0],
       [   5,    5,   25,    0,    0,   16,    0,   30,   46,    1,    0,
           0,    0,    6,    0,    0,    0,    0,    1,    0,    3,    9,
           1,    0,    5,    0,    2],
       [   3,    0,    3,    7,    0,   27,    0,   12,   22,    5,    0,
           0,    0,    3,    0,    0,    0,    0,    0,    0,    6,   12,
           0,    0,    0,    0,    0],
       [   0,    0,    0,    0,    0,    4,    1,   33,   10,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    3,
           0,    0,    2,    0,    0],
       [   6,    0,    1,    0,    0,  889,    2,   96,  425,    

In [ ]:
def predict_genre(plot):

    cleaned_plot = clean_text(plot)

    vectorized_plot = tfidf.transform([cleaned_plot])

    prediction = model.predict(vectorized_plot)

    return prediction[0]

In [ ]:
while True:

    plot = input("Enter movie plot (or type 'exit'): ")

    if plot.lower() == "exit":
        break

    genre = predict_genre(plot)

    print("Predicted Genre:", genre)
    print("-" * 50)

Enter movie plot (or type 'exit'): A brave soldier fights terrorists to save his country.
Predicted Genre: action
--------------------------------------------------
Enter movie plot (or type 'exit'): A family is haunted by a ghost in an old house.
Predicted Genre: horror
--------------------------------------------------
